[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/cyneuro/Chapter_Colabs/blob/main/Colab_C.ipynb)

# Set C — Cellular Models & Electrophysiology Basics
**Author: Neural Engineering Laboratory, University of Missouri-Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair**

## Introduction
Welcome to the Cellular Modeling module. This lab is designed to move you from a qualitative understanding of neurons to a **quantitative appreciation** of how they function. We achieve this by exploring the spectrum of neural abstraction: from the computationally efficient **LIF** and **Izhikevich** models used in large-scale networks, to the biophysically detailed **Hodgkin-Huxley (HH)** equations that simulate actual ion channel kinetics.

By the end of this module, you will be able to map biological phenomena (like refractory periods and thresholding) to specific mathematical parameters and observe how measured physical signals can be used to drive simulated neural activity.

> ### ⚠️ Critical Path for Learners
> While the notebook begins with simplified models (C1-C2), these abstractions assume a firm grasp of electrophysiology principles. **It is strongly recommended that you complete the [B-Appendix: Electrophysiology Basics](#B-Appendix) PRIOR to beginning the contents in Set B.** The Appendix acts as your "Experimental Bench," where you will learn how to measure input resistance ($R_{in}$) and time constants ($\tau$) — concepts essential for interpreting the models in the main set.

## Table of Contents
* **[Introduction](#Introduction)** — Module overview and goals.
* **[🔰 C0 Starter](#C0-Starter)** — Environment setup (Run this first!).
* **[B-Appendix: Electrophysiology Basics](#B-Appendix)** — **Start Here** to learn the "Experimental Mindset."
* **[C1 Leaky Integrate-and-Fire (LIF)](#C1-LIF)** — The fundamental RC integrator.
* **[C2 Izhikevich Model](#C2-Izhikevich)** — Adding adaptation and bursting with low cost.
* **[C3 Hodgkin-Huxley via NEURON](#C3-HH)** — Detailed channel kinetics and biophysics.
* **[C4 Real-world Data Bridge](#C4-microbit)** — Mapping sensor data (micro:bit) to current clamp.
* **[Reflection & Discussion](#Reflection)** — Synthesizing biology-to-model mapping.

## 🔰 C0 Starter — Setup & Readiness Check
Before we begin, we must initialize the programming environment. This cell installs the **NEURON** simulator and ensures the necessary libraries are ready for the Appendix and C3.

### 🛠️ Interactive Module Note
This notebook is designed as an **Interactive Experimental Rig**.
* **Sliders:** Use the sliders on the right of code cells to adjust parameters.
* **Auto-Run:** Most cells are set to `run: "auto"`, meaning the plots will update the moment you move a slider.

In [ ]:
# Install dependencies with quotes around the version constraint
!pip install -q neuron "numpy<2.1" matplotlib --upgrade

import neuron
from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# Ensure standard run functions are available
h.load_file('stdrun.hoc')

print("NEURON is ready and functional.")


## C1 — Leaky Integrate-and-Fire (LIF)
### 🎯 Interactive Exercise: The Temporal Integrator
The LIF model is a "leaky" bucket for charge. Use the sliders to find the balance between how fast the cell leaks ($tau$) and how much it resists change ($R$).

**Task:** Adjust the **tau** slider to 0.040 (40ms). Does the spike count increase or decrease compared to the baseline? Why?

---
### 🎯 Exercises: The Temporal Integrator
**Exercise 1: Time Constant Impact**
1. **Predict:** If you double the time constant ($\tau$) from 20ms to 40ms, will the neuron require a *higher* or *lower* frequency of input pulses to reach the threshold?
2. **Verify:** Change `tau = 40e-3` in the code cell above. Observe the "Spikes" count in the output. Did it change as you expected?

**Exercise 2: Resistance vs. Threshold**
1. **Predict:** If you increase the Input Resistance ($R$) from 100 to 150, will the "Spikes" count increase or decrease?
2. **Verify:** Adjust $R$ to 150 and run the cell. How does the voltage slope change during the stimulus?
---

In [ ]:
#@title LIF Experimental Rig { run: "auto" }
import numpy as np
import matplotlib.pyplot as plt

# Parameters from sliders
R = 100 #@param {type:"slider", min:50, max:250, step:5}
tau = 0.02 #@param {type:"slider", min:0.005, max:0.1, step:0.001}
V_rest, V_th, V_reset = -65, -50, -70

t = np.arange(0, 1.0, 1e-4)
I = np.zeros_like(t)
I[(t>0.1)&(t<0.6)] = 0.5; I[(t>0.7)&(t<0.8)] = 0.8
V = np.ones_like(t)*V_rest; spikes = []

for k in range(1, len(t)):
    V[k] = V[k-1] + ((-(V[k-1]-V_rest) + R*I[k-1])*(1e-4/tau))
    if V[k] >= V_th:
        spikes.append(t[k]); V[k] = V[k-1] = V_reset

plt.figure(figsize=(8, 3.5))
plt.plot(t, V, 'k'); plt.ylabel('V (mV)')
plt.twinx(); plt.plot(t, I, 'C0', alpha=0.3); plt.ylabel('I (nA)')
plt.title(f'LIF Response | Spikes: {len(spikes)}')
plt.show()

## C2 — Izhikevich Model
### 🎯 Exercise: Tuning Firing Patterns
This model uses four parameters ($a, b, c, d$) to replicate complex biology.
* **Regular Spiking (RS):** Use the default slider values.
* **Bursting:** Set **c** to -50 and **d** to 2.

---
### 🎯 Exercises: Adaptation & Bursting
**Exercise 1: Spike Frequency Adaptation**
1. **Predict:** Using the **Regular Spiking (RS)** parameters, will the interval between the 1st and 2nd spike be the same as the interval between the 5th and 6th?
2. **Verify:** Run the simulation. Use the plot to measure the Inter-Spike Interval (ISI) between early spikes versus late spikes.

**Exercise 2: Parameter Sensitivity**
1. **Predict:** The variable $d$ represents the "reset" of the recovery variable $u$. If you decrease $d$ from 8 to 2, will the neuron's firing rate increase or decrease?
2. **Verify:** Change $d=2$ in the RS parameters and run the cell.
3. **Challenge:** Change the parameters to $a=0.02, b=0.2, c=-50, d=2$ (**Bursting**). Identify the time length of the quiet period between bursts.
---

In [ ]:
#@title Izhikevich Parameter Tuning { run: "auto" }
import numpy as np
import matplotlib.pyplot as plt

a = 0.02 #@param {type:"slider", min:0.01, max:0.1, step:0.01}
b = 0.2 #@param {type:"slider", min:0.05, max:0.3, step:0.01}
c = -65 #@param {type:"slider", min:-75, max:-45, step:1}
d = 8 #@param {type:"slider", min:0.05, max:10.0, step:0.05}

T = 1000; I = np.zeros(T); I[200:800] = 10
v = -65 * np.ones(T); u = b * v

for k in range(1, T):
    dv = 0.04*v[k-1]**2 + 5*v[k-1] + 140 - u[k-1] + I[k-1]
    du = a*(b*v[k-1] - u[k-1])
    v[k] = v[k-1] + dv; u[k] = u[k-1] + du
    if v[k] >= 30:
        v[k-1]=30; v[k]=c; u[k]+=d

plt.figure(figsize=(8, 3.5)); plt.plot(v, 'k')
plt.title('C2: Interactive Izhikevich'); plt.ylabel('v (mV)'); plt.show()

## C3 — HH via NEURON

### Model Idea

Biophysical $Na^+$ / $K^+$ conductances produce threshold and refractoriness from channel kinetics; waveforms reflect identifiable mechanisms.

### 🎯 Interactive Exercise: Biophysical Kinetics
Use the sliders in the rig below to explore how ionic conductances dictate spike timing and recovery.

**Exercise 1: Kinetic Latency & Timing**
* **The Goal:** Observe how stimulus strength affects the "time-to-fire."
* **Task:** Move the `stim_delay` to 100ms. Now, slowly increase the `stim_amp` from 0.08 to 0.15 nA.
* **Observe:** Does the first spike move closer to the dotted blue "Onset" line? This represents the time required for sodium channels to reach the open-state threshold.

**Exercise 2: The Refractory Barrier & AHP**
* **The Goal:** Understand Afterhyperpolarization (AHP).
* **Observe:** Look at the "dip" in voltage immediately following a spike.
* **Task:** Adjust `stim_amp` to a high value (e.g., 0.25 nA) to cause rapid firing.
* **Verify:** Note how the cell cannot fire again until the voltage recovers from that dip. This is where $K^+$ gates are still closing, creating the refractory period you measured in the Appendix.

In [ ]:
#@title C3: Hodgkin-Huxley Experimental Rig { run: "auto" }
from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# 1. Clear memory first
h('forall delete_section()')

# 2. Parameters from Sliders
stim_amp = 0.1 #@param {type:"slider", min:0.05, max:0.3, step:0.01}
stim_dur = 600 #@param {type:"slider", min:100, max:800, step:50}
stim_delay = 100 #@param {type:"slider", min:10, max:300, step:10}

# 3. Rig Setup
soma = h.Section(name='soma_hh')
soma.L = soma.diam = 12.6157
soma.insert('hh')

stim = h.IClamp(soma(0.5))
stim.delay = stim_delay
stim.dur = stim_dur
stim.amp = stim_amp

# 4. Recording & Run
v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)
h.finitialize(-65)
h.continuerun(800)

# 5. Plotting
plt.figure(figsize=(8, 4))
plt.plot(t, v, 'k', lw=1.5, label='Membrane Voltage')
plt.axhline(-65, color='gray', linestyle='--', alpha=0.5)
plt.axvline(stim_delay, color='C0', linestyle=':', alpha=0.8, label='Stimulus Onset')
plt.title(f'HH Current Clamp | Delay: {stim_delay}ms | Amp: {stim_amp}nA')
plt.xlabel('Time (ms)'); plt.ylabel('Membrane Voltage (mV)')
plt.legend(loc='upper right'); plt.grid(alpha=0.2)
plt.show()

## C4 — Real-world Data Bridge

### Model Idea

In a Brain-Computer Interface (BCI) or sensory prosthetic, we don't use sliders—we use **Sensors**. However, sensors often output "Raw Units" (like 0-1023) that don't match the "Biophysical Units" (nA) of our model.

### 🎯 Interactive Exercise: The Scaling Challenge
**The Goal:** Map raw sensor data to the `stim.amp` of your HH model to trigger meaningful spikes.

1. **Observe the Input:** The top (gray) plot shows a signal recorded from an external device.
2. **Adjust `data_scale`:** This acts as your "Gain." If it's too low, the signal won't reach the **Rheobase** you discovered in the Appendix.
3. **Adjust `data_offset`:** This moves the entire signal up or down. Use this to set the "Baseline" firing rate.

**Exercise: Precision Triggering**
* **Task:** Adjust the sliders so that the neuron fires **exactly one spike** during the highest peak of the sensor data, but remains silent during the smaller ripples.
* **Reflect:** In a prosthetic limb, why is it critical to tune the `data_offset` so the model doesn't "fire" when the sensor is just reading background noise?

In [ ]:
#@title C4: Sensor-to-Soma Data Bridge { run: "auto" }
data_scale = 0.005 #@param {type:"slider", min:0.001, max:0.02, step:0.001}
data_offset = 0.05 #@param {type:"slider", min:-0.1, max:0.2, step:0.01}

from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# 1. Create a dummy "Real-world" sensor signal (e.g., an Accelerometer)
t_data = np.linspace(0, 500, 1000)
raw_sensor = 50 + 30 * np.sin(2 * np.pi * 0.01 * t_data) + 10 * np.random.randn(1000)

# 2. Map Sensor -> Model Amps
# We apply the user's scaling and offset here
mapped_amps = (raw_sensor * data_scale) + data_offset

# 3. Setup NEURON Rig
h('forall delete_section()')
soma_c4 = h.Section(name='somac4')
soma_c4.L = soma_c4.diam = 20
soma_c4.insert('hh')

# Use play() to drive the injection with the mapped array
stim_vec = h.Vector(mapped_amps)
t_vec_stim = h.Vector(t_data)
stim = h.IClamp(soma_c4(0.5))
stim.delay = 0
stim.dur = 1e9 # Keep open to follow the vector
stim_vec.play(stim._ref_amp, t_vec_stim, 1)

# Record
v = h.Vector().record(soma_c4(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

h.finitialize(-65)
h.continuerun(500)

# 4. Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax1.plot(t_data, raw_sensor, color='gray', alpha=0.5)
ax1.set_ylabel('Raw Sensor Units')
ax1.set_title('External Data Input')

ax2.plot(t, v, 'k')
ax2.set_ylabel('Membrane V (mV)')
ax2.set_xlabel('Time (ms)')
ax2.axhline(-50, color='red', ls=':', alpha=0.5, label='Threshold')

plt.tight_layout()
plt.show()

## B-Appendix: Electrophysiology Basics

This appendix serves as your "Experimental Bench." While Sections C1–C4 focused on high-level abstractions, these exercises use the **NEURON environment** to explore the biophysical reality of how we measure and manipulate neurons.

## 🔬 The Experimental Mindset
In this section, you are the electrophysiologist. For each module (B-A.1 through B-A.6), you will find:
* ### Model Idea
  The biological or physical concept we are testing.
* ### Useful Checks
  Specific signal features you should look for in the plots to ensure the "rig" is working correctly.
* ### Predict → Verify
  A guided experiment where you must guess the outcome before running the code.

## 🛠️ Strategic Lab Guide
As you work through these basics, keep these "Troubleshooting" rules in mind for your later research:
1. **Too much "lag"?** If EPSPs are too broad in your models, revisit the **$\tau$** (Time Constant) measurements in **B-A.4**.
2. **Missing Spikes?** If a cell won't fire, check your **Rheobase** findings in **B-A.6**.
3. **Wrong Direction?** If inhibition is causing depolarization, check **$E_{rev}$** (Reversal Potential) in **B-A.5**.

*Student Note: Ensure you have run the **C0 Starter** cell at the top of this notebook, or the NEURON-based simulations in this appendix will not initialize.*

## B-A.1 — The "Zero-Point" and Resting Potential
### Model Idea

An electrode measures the potential difference between the salty interior of the cell and the grounded exterior. If we "force" a cell to a specific voltage and then let go, it will naturally drift toward its **Resting Potential** (governed by its leak channels, $e\_pas$).

### 🎯 Interactive Exercise: The Drift Test
**The Goal:** Observe how the membrane "relaxes" back to equilibrium.
1. **Adjust the `v_init` slider.** This simulates "clamping" the cell to a specific voltage before releasing it.
2. **Observe:** No matter where you start ($-80$ or $-50$ mV), the cell always wants to return to the same baseline. This baseline is the **Electrochemical Equilibrium**.

In [ ]:
#@title B-A.1: Resting Potential Drift { run: "auto" }
v_init = -75 #@param {type:"slider", min:-90, max:-40, step:1}

# Setup
h('forall delete_section()')
soma_a1 = h.Section(name='soma_a1')
soma_a1.insert('pas')
soma_a1.e_pas = -65  # This is the target "Rest"

# Record
v = h.Vector().record(soma_a1(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

# Run
h.finitialize(v_init)
h.continuerun(40)

# Plot
plt.figure(figsize=(8, 3.5))
plt.plot(t, v, 'k', lw=2)
plt.axhline(-65, color='r', linestyle='--', alpha=0.6, label='Leak Potential (e_pas)')
plt.ylim(-95, -35)
plt.title(f"Initial V: {v_init}mV | Drift toward Equilibrium")
plt.xlabel("Time (ms)"); plt.ylabel("V (mV)"); plt.legend()
plt.show()

## B-A.2 — Current Clamp & Rheobase
### Model Idea

Current clamp is like a "Volume Control" for charge. You inject a fixed amount of current ($I$), and the membrane voltage ($V_m$) responds based on Ohm's Law ($V = IR$).

### 🎯 Interactive Exercise: Finding the Rheobase
**The Goal:** Find the minimum current needed to trigger a single action potential.
1. **Slowly increase the `stim_amp` slider.** 2. **Identify the transition:** At what exact nA value does the "passive bump" suddenly turn into a full-blown spike? This value is the **Rheobase**.

In [ ]:
#@title B-A.2: The Current Clamp Rig { run: "auto" }
stim_amp = 0.02 # @param {"type":"slider","min":0,"max":0.1,"step":0.01}

# Setup
h('forall delete_section()')
soma_a2 = h.Section(name='soma_a2')
soma_a2.L = soma_a2.diam = 20
soma_a2.insert('hh')

stim = h.IClamp(soma_a2(0.5))
stim.delay = 10
stim.dur = 50
stim.amp = stim_amp

v = h.Vector().record(soma_a2(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

h.finitialize(-65)
h.continuerun(100)

plt.figure(figsize=(8, 3.5))
plt.plot(t, v, 'k')
plt.title(f"Current Injection: {stim_amp} nA")
plt.xlabel("ms"); plt.ylabel("mV")
plt.axhline(-50, color='gray', linestyle=':', label='Approx. Threshold')
plt.grid(alpha=0.2); plt.legend(); plt.show()

## B-A.3 — Voltage Clamp: The "Thermostat"

### Model Idea

In **Voltage Clamp**, we flip the script. Instead of injecting current to see the voltage change, we *force* the voltage to a specific value and measure the current required to keep it there. This "Current Flux" tells us exactly what the ion channels are doing.

### 🎯 Interactive Exercise: Command & Control
**The Goal:** Observe the $Na^+$ and $K^+$ currents hidden inside the membrane.

1. **Move the `v_step` slider.** Watch the orange current trace ($I$).
2. **Find the Inward Flux:** Set `v_step` to -20mV. Notice the sharp downward dip? That's Sodium ($Na^+$) rushing *into* the cell.
3. **Observe the Outward Flux:** Notice how the current stays high after the dip? That's Potassium ($K^+$) rushing *out*.
4. **The Inversion:** Move the slider to +50mV. Does the sodium "dip" disappear or change direction?

In [ ]:
#@title B-A.3: Voltage Clamp Rig { run: "auto" }
v_step = -20 #@param {type:"slider", min:-90, max:50, step:2}

# Setup
from neuron import h
import numpy as np
import matplotlib.pyplot as plt

h('forall delete_section()')
soma_a3 = h.Section(name='soma_a3')
soma_a3.L = soma_a3.diam = 20
soma_a3.insert('hh')

# V-Clamp Protocol: 3 stages
vcl = h.VClamp(soma_a3(0.5))
vcl.dur[0] = 10; vcl.amp[0] = -80  # Baseline
vcl.dur[1] = 25; vcl.amp[1] = v_step # The Step
vcl.dur[2] = 15; vcl.amp[2] = -80  # Return

# Record
i = h.Vector().record(vcl._ref_i)
v = h.Vector().record(soma_a3(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

h.finitialize(-65)
h.continuerun(50)

# Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Top plot: Voltage Command
ax1.plot(t, v, 'k', lw=2)
ax1.set_ylabel('V (mV)')
ax1.set_title(f'Voltage Clamp: Commanded Step to {v_step} mV')
ax1.grid(alpha=0.2)

# Bottom plot: Measured Current
ax2.plot(t, i, 'C1', lw=2, label='Clamp Current (I)')
ax2.set_ylabel('I (nA)')
ax2.set_xlabel('Time (ms)')
ax2.axhline(0, color='black', lw=1, ls='--')
ax2.legend()
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## B-A.4 — Estimating $R_{in}$ and $\tau$

### Model Idea

We use small, "Sub-threshold" pulses to characterize the physical "health" and size of the cell without triggering spikes. Two key numbers define this:
* **Input Resistance ($R_{in}$):** How "leaky" is the membrane? (Measured in MΩ)
* **Time Constant ($\tau$):** How "sluggish" is the membrane? (Measured in ms)

### 🎯 Interactive Exercise: The Passive Membrane
**The Goal:** Observe how changing the cell's physical size transforms its electrical identity.

1. **Increase the `diameter` slider.** Observe the voltage "bump" on the plot.
2. **Predict:** Does a larger cell (more surface area) have a higher or lower resistance ($R_{in}$)?
3. **Verify:** Use the slider to confirm. Notice how the calculated $R_{in}$ and $\tau$ values in the title change instantly.
*Hint: A larger cell is like a larger bucket with more holes—it is harder to fill up and leaks faster!*

In [ ]:
#@title B-A.4: Resistance & Time Constant Rig { run: "auto" }
diameter = 20 #@param {type:"slider", min:10, max:100, step:5}

from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# Setup
h('forall delete_section()')
soma_a4 = h.Section(name='soma_a4')
soma_a4.L = soma_a4.diam = diameter
soma_a4.insert('pas')
soma_a4.g_pas = 0.0002  # Leak conductance (S/cm2)
soma_a4.e_pas = -65     # Leak reversal (mV)

# Sub-threshold pulse (-0.05 nA)
stim = h.IClamp(soma_a4(0.5))
stim.amp = -0.05
stim.delay = 10
stim.dur = 50

# Record
v = h.Vector().record(soma_a4(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

h.finitialize(-65)
h.continuerun(100)

# Automatic Calculations
vnp = np.array(v)
tnp = np.array(t)
base_v = -65
steady_v = vnp[(tnp > 50) & (tnp < 60)].mean()
dV = steady_v - base_v
Rin = abs(dV) / abs(stim.amp) # R = V / I

# Find Tau (time to reach 63.2% of steady state)
target_v = base_v + 0.632 * dV
idx = np.where(vnp <= target_v)[0]
tau = (tnp[idx[0]] - 10) if len(idx) > 0 else 0

# Plotting
plt.figure(figsize=(8, 4))
plt.plot(tnp, vnp, 'k', lw=2)
plt.axhline(steady_v, color='C0', linestyle='--', alpha=0.5, label='Steady State')
plt.title(f"Diameter: {diameter}µm | Rin: {Rin:.1f} MΩ | τ: {tau:.2f} ms")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.grid(alpha=0.2)
plt.show()

## B-A.5 — I–V Curve & Reversal Potential

### Model Idea

By plotting steady-state current ($I$) against commanded voltage ($V$), we create an **I-V Curve**. The zero-crossing point ($I=0$) is the **Reversal Potential** ($E_{rev}$)—the voltage where no net ions flow.

### 🎯 Interactive Exercise: Probing the Membrane
**The Goal:** Find the voltage where the cell is "electrically silent."
1. **Adjust the `cmd_v` slider.** Watch the orange dot move along the blue I-V curve.
2. **Find the Zero-Crossing:** Move the slider until the "Current Vector" (orange dot) sits exactly on the horizontal black line ($I=0$).
3. **Question:** Based on the `hh` mechanism, is this value closer to -65mV or 0mV?

In [ ]:
#@title B-A.5: Interactive I-V Probe { run: "auto" }
cmd_v = -65 #@param {type:"slider", min:-90, max:50, step:1}

h('forall delete_section()')
soma_iv = h.Section(name='soma_iv')
soma_iv.L = soma_iv.diam = 20
soma_iv.insert('hh')

vcl = h.VClamp(soma_iv(0.5))
vcl.dur[0] = 5; vcl.amp[0] = -80
vcl.dur[1] = 30; vcl.amp[1] = cmd_v
vcl.dur[2] = 5; vcl.amp[2] = -80

i_vec = h.Vector().record(vcl._ref_i)
t_vec = h.Vector().record(h._ref_t)
h.finitialize(-65); h.continuerun(40)

# Extract steady-state current
current_at_v = np.array(i_vec)[(np.array(t_vec) >= 30) & (np.array(t_vec) <= 34)].mean()

# Reference background I-V curve
v_range = np.arange(-90, 51, 10)
i_range = [-0.19, -0.07, 0.04, 0.16, 0.28, 0.40, 1.2, 3.5, 7.2, 12.0, 18.0, 25.0, 33.0, 42.0, 52.0]

plt.figure(figsize=(6, 4))
plt.plot(v_range, i_range, 'b-', alpha=0.3, label='Full I-V Curve')
plt.axhline(0, color='k', lw=1)
plt.plot(cmd_v, current_at_v, 'C1o', markersize=10, label='Current Vector')
plt.xlabel('Command V (mV)'); plt.ylabel('Steady I (nA)')
plt.title(f'I-V Probe: V = {cmd_v} mV | I = {current_at_v:.2f} nA')
plt.grid(alpha=0.2); plt.legend(); plt.show()

## B-A.6 — The Refractory Barrier
### Model Idea
After a spike, a neuron enters a "recovery" phase where it is harder—or impossible—to fire again.

**Task: Identify the Absolute Refractory Period**
1. Set the **isi** (Inter-Stimulus Interval) to 15ms. You should see two distinct spikes.
2. Slowly decrease the **isi**. At what value does the second spike completely disappear? This is the **Absolute Refractory Period**.
3. **Challenge:** If you increase `p2_amp`, can you force a spike to happen earlier? (This explores the **Relative Refractory Period**).

In [ ]:
#@title B-A.6: Paired-Pulse Interaction { run: "auto" }
isi = 12 #@param {type:"slider", min:1, max:20, step:0.5}
p2_amp = 0.35 #@param {type:"slider", min:0.1, max:1.0, step:0.01}

h('forall delete_section()')
soma_ref = h.Section(name='soma_ref')
soma_ref.L = soma_ref.diam = 20
soma_ref.insert('hh')

stim1 = h.IClamp(soma_ref(0.5))
stim1.delay = 5; stim1.dur = 2; stim1.amp = 0.5

stim2 = h.IClamp(soma_ref(0.5))
stim2.delay = 5 + isi; stim2.dur = 2; stim2.amp = p2_amp

v = h.Vector().record(soma_ref(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

h.finitialize(-65); h.continuerun(35)
plt.figure(figsize=(8, 3.5)); plt.plot(t, v, 'k')
plt.axvline(5 + isi, color='C1', linestyle=':', alpha=0.5, label='Pulse 2 Onset')
plt.title(f"Refractory Test | ISI: {isi}ms | Pulse 2 Amp: {p2_amp}nA")
plt.xlabel("Time (ms)"); plt.ylabel("Voltage (mV)")
plt.legend(); plt.show()

## Reflection & Synthesis

### Model Matching
Which model (**LIF** vs **Izhikevich**) allowed you to best replicate the specific firing patterns you observed in the **B-Appendix**? Justify your choice by comparing the flexibility of the sliders (parameters) available in each model.

### Parametric Mapping: From Measurement to Model
In the Appendix (**B-A.4**), you used a slider to see how `diameter` changes $\tau$ and $R_{in}$.
* How did the $R_{in}$ you measured in the Appendix manifest when you were tuning the **LIF Experimental Rig**?
* If a cell has a very high $R_{in}$, would you expect to set the `stim_amp` slider higher or lower to achieve a spike?

## Practice / Discussion Questions — Set C — Biology → Model Mapping

1. **Model Abstraction:** Define a "model abstraction" for a neuron that preserves interpretability. Which parameters must stay in **biophysical units** (e.g., mV, nA), and why?
2. **The Power of the Clamp:** In **B-A.3**, you used a **Voltage Clamp slider** to "unmask" $Na^+$ and $K^+$ currents. Why is it impossible to see these individual currents clearly in a standard Current Clamp?
3. **Geometric Scaling:** Based on your exploration in **B-A.4**, if you modeled a large Alpha Motor Neuron (large diameter), how would you adjust the `R` and `tau` sliders in the **LIF model** to stay accurate?
4. **Rheobase Discovery:** In **B-A.2**, you found the **Rheobase**. If you increased the "leakiness" (`g_pas`) of that cell, would you expect the Rheobase to increase or decrease?
5. **Efficiency vs. Detail:** *Justify:* What is gained and lost when moving from a detailed **HH-type model** (C3) to a reduced **Izhikevich model** (C2)?
6. **Refractoriness:** Compare the "Refractory" behavior in **C2 (Izhikevich)** vs **B-A.6 (HH)**. Which model felt more "biological" as you moved the ISI slider?
7. **The Data Bridge:** In **C4**, we map sensor values to `stim.amp`. If your sensor data is "noisy," how might that noise interact with the **Rheobase** threshold?
8. **Parameter Transparency:** Why is "parameter transparency" essential for educational modeling? Give one consequence of using "opaque" scaling (e.g., arbitrary units).
9. **I-V Linearity:** In **B-A.5**, is the I-V curve a perfectly straight line? What biological feature causes it to curve at higher voltages?
10. **Time Constant Intuition:** If two neurons have the same **Input Resistance** but different **Capacitances**, which one will respond faster to a sudden slider change in `stim_amp`?
11. **Reset Dynamics:** In the **Izhikevich model**, the `c` slider sets the reset voltage. How does this mimic the **Afterhyperpolarization (AHP)** you observed in the HH model?
12. **The Role of NEURON:** What is the primary advantage of using a simulator like **NEURON** (C3) over writing your own Euler integration (C1) from scratch?
13. **Assumptions:** How do you **document assumptions** so a reader can reproduce and critique your model? (List 3-4 items).
14. **Validation:** How would you **validate** that your interactive rig is working correctly before using it to collect data for a lab report?
15. **Missing Mechanisms:** Give an example of a **missing mechanism** (e.g., omitting Calcium) that could lead to a consistent mismatch between your model and real data.
16. **Bursting Logic:** In **C2**, you tuned the `d` parameter for bursting. Biologically, what might `d` represent in terms of slow ion channel recovery?
17. **Step Tests:** Why do we advocate for a **"membrane step test"** (B-A.4) as the first step in characterizing any new neural model?
18. **Reproducible Workflows:** Outline a minimal **reproducible workflow** (notebook → plots → metrics) for moving from an Appendix measurement to a Main Set simulation.
19. **Educational Trade-offs:** What is the trade-off of adding **noise sources** to a beginner-level model? Does it help or hinder the learning of core concepts?
20. **Forward Reflection:** What core principle from **Set B** (Cellular) will be most important when you begin analyzing **spatial effects** (Dendrites/Axons) in Set C?